# Playbook 04 — Knowledge-Graph Embedding Training

Train RotatE / ComplEx entity embeddings on Hetionet, save them to `data/`, and inspect the geometry.

**Inputs:**
- `data/hetionet-v1.0-edges.sif`
- `data/hetionet-v1.0-nodes.tsv`
- `config/kg_layer_config.yaml` (optional)

**Outputs:**
- `data/entity_embeddings.npy` — N × D full-dimensional embeddings
- `data/entity_ids.json` — entity → row index mapping
- `data/complex_128d_entity_embeddings.npy` — pre-trained 128d ComplEx checkpoint (if present)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
from pathlib import Path
import numpy as np
import pandas as pd

## 1. Load Hetionet edges (CtD task focus)

The pipeline trains over a single relation type at a time. CtD (Compound-treats-Disease) is the headline benchmark used for the PR-AUC 0.7987 result.

In [ ]:
from kg_layer.kg_loader import load_hetionet_edges, extract_task_edges, prepare_full_graph_for_embeddings

edges_path = Path('data/hetionet-v1.0-edges.sif')
if not edges_path.exists():
    raise FileNotFoundError(
        f'Missing {edges_path}. Download with: python -m kg_layer.kg_loader --download'
    )

edges_df = load_hetionet_edges(str(edges_path))
print(f'Total edges: {len(edges_df):,}')
print(f'Relation types: {edges_df["metaedge"].nunique() if "metaedge" in edges_df.columns else "?"}')

In [ ]:
# Task subset: Compound-treats-Disease
task_edges = extract_task_edges(edges_df, relation='CtD')
print(f'CtD edges: {len(task_edges):,}')
task_edges.head()

## 2. Prepare full-graph triple set for the embedder

RotatE / ComplEx are trained on the **whole** graph — the task subset only matters for the downstream classifier. This gives the embeddings access to all relational context (drug-target, gene-pathway, etc.).

In [ ]:
triples_df = prepare_full_graph_for_embeddings(edges_df)
print(f'Triples for training: {len(triples_df):,}')
print(f'Unique entities: {len(set(triples_df["head"]) | set(triples_df["tail"])):,}')
print(f'Unique relations: {triples_df["relation"].nunique()}')

## 3. Train embeddings

Pin to RotatE + 32d (the headline configuration). The locked values live in `utils/preregistered_constants.py`:

- `EMBEDDING_METHOD = "RotatE"`
- `EMBEDDING_DIM = 32`
- `EMBEDDING_EPOCHS = 200`

If `pykeen` is installed, the embedder uses it; otherwise it falls back to a deterministic embedding for testing. Training takes ~20-40 min on CPU and ~2-5 min on DGX GPU.

In [ ]:
from kg_layer.kg_embedder import HetionetEmbedder
from utils.preregistered_constants import EMBEDDING_DIM, EMBEDDING_EPOCHS, EMBEDDING_METHOD

print(f'Method: {EMBEDDING_METHOD}, dim: {EMBEDDING_DIM}, epochs: {EMBEDDING_EPOCHS}')

embedder = HetionetEmbedder(embedding_dim=EMBEDDING_DIM, qml_dim=5, work_dir='data')

In [ ]:
# Use cached embeddings if present (saves training time)
if embedder.load_saved_embeddings(expected_dim=EMBEDDING_DIM):
    print(f'Loaded cached embeddings: {embedder.entity_embeddings.shape}')
else:
    print('No cache hit — training from scratch (this may take a while)')
    embedder.train_embeddings(triples_df)
    print(f'Trained embeddings: {embedder.entity_embeddings.shape}')

## 4. Reduce to QML dimensionality (PCA → qml_dim)

The QSVC operates on 5–8 qubits; the full 32d embedding is reduced via PCA so the quantum feature map can encode each entity in available qubits.

In [ ]:
embedder.reduce_to_qml_dim()
print(f'Reduced shape: {embedder.reduced_embeddings.shape}')
print(f'Explained variance (5 components): '
      f'{np.cumsum(embedder._pca.explained_variance_ratio_)[-1]:.3f}'
      if embedder._pca is not None else '(deterministic mode)')

## 5. Inspect embedding geometry

In [ ]:
# Quick sanity: vector norms and pairwise distances of a few well-known compounds
from entity_resolution.hetionet_resolver import HetionetResolver
resolver = HetionetResolver().load()

probes = ['metformin', 'aspirin', 'imatinib', 'tamoxifen', 'warfarin']
vecs = {}
for name in probes:
    hid = resolver.resolve(name)
    if hid and hid in embedder.entity_to_id:
        idx = embedder.entity_to_id[hid]
        vecs[name] = embedder.entity_embeddings[idx]

for name, v in vecs.items():
    print(f'  {name:12s}  |v|={np.linalg.norm(v):.3f}')

In [ ]:
# Pairwise cosine similarity
names = list(vecs.keys())
mat = np.array([vecs[n] for n in names])
norms = np.linalg.norm(mat, axis=1, keepdims=True)
cos = (mat @ mat.T) / (norms @ norms.T + 1e-12)
pd.DataFrame(np.round(cos, 3), index=names, columns=names)

## 6. Export for downstream layers

Already done by `train_embeddings()` — files land in `data/`:
- `entity_embeddings.npy` (N × 32)
- `entity_ids.json` ({entity_id: row_index})

**Next:** [`05_hybrid_qml_prediction.ipynb`](05_hybrid_qml_prediction.ipynb) consumes these embeddings to build QSVC + classical ensemble predictions.

In [ ]:
for p in ['data/entity_embeddings.npy', 'data/entity_ids.json']:
    f = Path(p)
    print(f'  {"✓" if f.exists() else "✗"}  {p}  ({f.stat().st_size if f.exists() else 0:,} bytes)')